# Cis4930 Project 1
## Liam Sharkey

In [ ]:
#setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast

#there's a lot of stuff in there i dont want
useful_cols = ['appid', 'name', 'release_date', 'price', 'dlc_count',
                'windows', 'mac', 'linux', 'metacritic_score',
               'achievements','recommendations','supported_languages',
               'full_audio_languages','developers','publishers','categories',
               'genres','score_rank','positive','negative','average_playtime_forever',
               'average_playtime_2weeks','median_playtime_forever',
               'median_playtime_2weeks','discount','peak_ccu','tags',
               'pct_pos_total','num_reviews_total','pct_pos_recent',
               'num_reviews_recent']

#had to specify python engine because c was having problems
df = pd.read_csv('data/raw/games_march2025_full.csv', 
        on_bad_lines='skip', engine='python', usecols=useful_cols)

#fill in missing data with average where available, and sentinel where not
df = df.fillna(df.mean(numeric_only=True))
string_columns = df.select_dtypes(include=['str']).columns
df[string_columns] = df[string_columns].fillna('MISSING')
pd.isnull(df).sum().sum()

#convert bools to ints
df['windows'] = df['windows'].astype(int)
df['mac'] = df['mac'].astype(int)
df['linux'] = df['linux'].astype(int)

#convert to proper lists
df["supported_languages"] = df["supported_languages"].apply(ast.literal_eval)
df["full_audio_languages"] = df['full_audio_languages'].apply(ast.literal_eval)
df['genres'] = df['genres'].apply(ast.literal_eval)

#convert release date to datetime obj
df['release_date'] = pd.to_datetime(df['release_date'])

#df.loc[df['pct_pos_total'].idxmax()]  #for ref

In [109]:
df.describe()

,appid,release_date,price,dlc_count,windows,mac,linux,metacritic_score,achievements,recommendations,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent,accessibility_score
count,9.040000e+04,90400,90400.000000,90400.000000,90400.000000,90400.000000,90400.000000,90400.000000,90400.000000,9.040000e+04,...,90400.000000,9.040000e+04,90400.000000,90400.000000,9.040000e+04,90400.000000,9.040000e+04,90400.000000,90400.000000,90400.000000
mean,1.698749e+06,2021-05-15 09:48:41.734513,6.953921,0.584480,0.999668,0.186947,0.135277,2.900819,20.124591,1.073639e+03,...,4.996659,1.138092e+02,5.270376,4.403197,9.752251e+01,46.925907,1.520945e+03,5.646338,17.779403,8.398894
min,2.000000e+01,1997-06-30 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,-1.000000,-1.000000e+00,-1.000000,-1.000000,1.000000
25%,8.751975e+05,2019-01-28 00:00:00,0.990000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,-1.000000,-1.000000e+00,-1.000000,-1.000000,2.000000
50%,1.583430e+06,2022-02-09 00:00:00,3.990000,0.000000,1.000000,0.000000,0.000000,0.000000,3.000000,0.000000e+00,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,62.000000,1.700000e+01,-1.000000,-1.000000,3.000000
75%,2.485098e+06,2024-01-19 00:00:00,9.990000,0.000000,1.000000,0.000000,0.000000,0.000000,20.000000,0.000000e+00,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,85.000000,9.000000e+01,-1.000000,-1.000000,7.000000
max,3.570420e+06,2025-03-10 00:00:00,999.980000,3427.000000,1.000000,1.000000,1.000000,97.000000,9821.000000,4.401572e+06,...,18568.000000,1.462997e+06,18568.000000,100.000000,1.212356e+06,100.000000,8.632939e+06,100.000000,96473.000000,209.000000
std,9.291075e+05,NaN,13.167653,15.274738,0.018214,0.389871,0.342021,14.443214,163.699897,2.330537e+04,...,180.329695,8.768545e+03,188.819105,16.282217,5.692771e+03,40.515959,3.636148e+04,22.972753,470.504368,22.737546


## How does accessibility effect a game

Support score represents the number of different types of people
who would be able to confortably and conveniently play the game

In [ ]:
df["accessibility_score"] = (df['windows'] + df['mac'] + df['linux'] + 
    df['supported_languages'].apply(len) + df['full_audio_languages'].apply(len))

In [ ]:
filtered_df = df[df['pct_pos_total'] > 0] #a user score of 0 implies no reviews
x = filtered_df['accessibility_score']
y = filtered_df['pct_pos_total']
plt.scatter(x, y)
plt.plot(np.unique(x), np.poly1d(np.polyfit(x, y, 1))(np.unique(x)), color='red')
plt.xlabel("Accessibility Score")
plt.ylabel("Rating")
#plt.xlim(0, 150)
#plt.ylim(20) #a user score of 0 indicates it just doesn't have reviews
plt.title("Correlation between accessibility and rating")
plt.show

## Trends within genre

In [ ]:
df_genres = df.explode('genres')
df_genres = df_genres[df_genres['pct_pos_total'] >= 0] #-1 doesn't mean anyhting
df_genres.groupby('genres')['pct_pos_total'].median()

genres
360 Video                 -1
Accounting                93
Action                   100
Adventure                100
Animation & Modeling      97
Audio Production          96
Casual                   100
Design & Illustration    100
Documentary               -1
Early Access             100
Education                100
Episodic                  -1
Free To Play             100
Game Development          97
Gore                     100
Indie                    100
Massively Multiplayer    100
Movie                     53
Nudity                    93
Photo Editing             -1
RPG                      100
Racing                   100
Sexual Content            96
Short                     -1
Simulation               100
Software Training         94
Sports                   100
Strategy                 100
Tutorial                  -1
Utilities                 92
Video Production          90
Violent                  100
Web Publishing            70
Name: pct_pos_total, dtype: int64